In [1]:
from huggingface_hub import login
login(new_session=False)

In [2]:
!git clone https://github.com/AI4Bharat/IndicTrans2.git

%cd /kaggle/working/IndicTrans2/huggingface_interface

Cloning into 'IndicTrans2'...
remote: Enumerating objects: 771, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 771 (delta 154), reused 106 (delta 104), pack-reused 585 (from 3)
Receiving objects: 100% (771/771), 4.16 MiB | 18.20 MiB/s, done.
Resolving deltas: 100% (496/496), done.
/kaggle/working/IndicTrans2/huggingface_interface


In [3]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers==4.53.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

In [4]:
import os
import csv
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from IndicTransToolkit.IndicTransToolkit import IndicProcessor
from tqdm import tqdm

2026-01-05 04:23:08.666018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767586988.932235      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767586989.005290      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767586989.634313      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767586989.634368      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767586989.634372      17 computation_placer.cc:177] computation placer alr

In [5]:
DATA_PATH = "/kaggle/input/pairs-lang-correct-corrupted-jumbled/pairs_lang_correct_corrupted_jumbled.csv"
OUTPUT_DIR = "/kaggle/working/parallel_corpus"

os.makedirs(OUTPUT_DIR, exist_ok=True)

test_path = os.path.join(OUTPUT_DIR, "test.txt")
with open(test_path, "w") as f:
    f.write("Kaggle write test")

print("Files in output dir:", os.listdir(OUTPUT_DIR))

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32          
MAX_LENGTH = 128
QUANTIZATION = None      

LANGS = [ "mar_Deva", "ben_Beng", "guj_Gujr", "asm_Beng", "ory_Orya", ]

Files in output dir: ['test.txt']


In [6]:
def load_model_and_tokenizer():
    qconfig = None
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, trust_remote_code=True
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    ).to(DEVICE)

    model.eval()
    return tokenizer, model

In [7]:
def batch_translate_generator(sentences, src_lang, tgt_lang, model, tokenizer, ip):
    for i in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[i:i + BATCH_SIZE]

        batch = ip.preprocess_batch(
            batch, src_lang=src_lang, tgt_lang=tgt_lang
        )

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_length=MAX_LENGTH,
                num_beams=1,
                do_sample=False,
                use_cache=True,
            )

        translations = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        translations = ip.postprocess_batch(
            translations, lang=tgt_lang
        )

        yield translations

In [8]:
def translate_language_pair(src_lang, tgt_lang, sentences, model, tokenizer, ip):
    out_path = os.path.join(
        OUTPUT_DIR, f"{src_lang}_to_{tgt_lang}.csv"
    )

    with open(out_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([src_lang, tgt_lang])

        idx = 0
        for batch_out in tqdm(
            batch_translate_generator(
                sentences, src_lang, tgt_lang, model, tokenizer, ip
            ),
            desc=f"{src_lang} → {tgt_lang}",
        ):
            for s, t in zip(
                sentences[idx:idx + len(batch_out)],
                batch_out
            ):
                writer.writerow([s, t])

            idx += len(batch_out)

    print(f"Saved → {out_path}")


In [9]:
def already_done(src_lang, tgt_lang):
    out_path = os.path.join(
        OUTPUT_DIR, f"{src_lang}_to_{tgt_lang}.csv"
    )
    return os.path.exists(out_path)


In [10]:
df = pd.read_csv(DATA_PATH)
df["language"] = df["language"].astype(str).str.strip()

tokenizer, model = load_model_and_tokenizer()
ip = IndicProcessor(inference=True)

for src_lang in LANGS:
    print(f"\nProcessing source language: {src_lang}")

    src_df = df[df["language"] == src_lang]
    sentences = src_df["correct"].astype(str).tolist()

    print(f"Sentences: {len(sentences)}")
    assert len(sentences) > 0, f"No data for {src_lang}"

    for tgt_lang in LANGS:
        if tgt_lang == src_lang:
            continue
    
        if already_done(src_lang, tgt_lang):
            print(f"Skipping existing: {src_lang} → {tgt_lang}")
            continue
    
        translate_language_pair(
            src_lang,
            tgt_lang,
            sentences,
            model,
            tokenizer,
            ip,
        )

    torch.cuda.empty_cache()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B.
401 Client Error. (Request ID: Root=1-695b3cc5-376ca349506b0af80fdc7023;5a582fe0-a29d-4df4-8f9b-6202000d82e3)

Cannot access gated repo for url https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B/resolve/main/config.json.
Access to model ai4bharat/indictrans2-indic-indic-1B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
import os

base_dir = "/kaggle/working/parallel_corpus"

files_to_delete = [
    "guj_Gujr_to_asm_Beng.csv",
    "ben_Beng_to_ory_Orya.csv",
    "asm_Beng_to_ory_Orya.csv",
]

for f in files_to_delete:
    path = os.path.join(base_dir, f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {f}")
    else:
        print(f"Not found: {f}")

In [ ]:
!mkdir dataset_upload

In [ ]:
!cp -r /kaggle/working/parallel_corpus dataset_upload/

In [ ]:
%%writefile dataset_upload/dataset-metadata.json
{
  "title": "Parallel Corpus Outputs",
  "id": "nakshsingh18/parallel-corpus-outputs",
  "licenses": [
    { "name": "CC0-1.0" }
  ]
}

In [ ]:
!mkdir -p /root/.config/kaggle

In [ ]:
%%writefile /root/.config/kaggle/kaggle.json
{
  "username": "nakshsingh",
  "key": "KGAT_85d29120257d79a9c9ee5a50d2df7f28"
}

In [ ]:
!chmod 600 /root/.config/kaggle/kaggle.json


In [ ]:
!ls -l /root/.config/kaggle/kaggle.json


In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets create -p dataset_upload --dir-mode zip